# 🫁 MedGemma Chest X-Ray Analyzer — Kaggle

Analyze chest X-ray images using **Google's MedGemma 1.5 (4B)** medical vision-language model.

---

## Setup Checklist
1. ✅ **Enable GPU**: Settings (right panel) → Accelerator → **GPU T4 x1** *(NOT P100 — P100 is incompatible with this model's required PyTorch version)*
2. ✅ **Add HuggingFace token**: Add-ons → Secrets → `HF_TOKEN` (toggle "Notebook access" ON)
3. ✅ **Accept MedGemma license**: [huggingface.co/google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it)
4. ✅ **Run All**: Run → Run All

---

> ⚠️ **Disclaimer**: For research purposes only. Not for clinical diagnosis.

In [ ]:
# CELL 1 — Install Required Packages
# We need to pin Pillow to an older version because Pillow 12 broke
# an internal function that torchvision depends on.
# PyTorch itself stays at Kaggle's default (2.6+) since MedGemma needs it.
# transformers  → loads and runs the MedGemma model
# accelerate    → helps run the model efficiently on GPU
# bitsandbytes  → enables 4-bit quantization to reduce memory usage
# requests      → used to download the sample X-ray image

!pip install "Pillow<12.0" -q
!pip install --upgrade --quiet transformers accelerate bitsandbytes requests

print('✅ All packages installed successfully')

In [ ]:
# CELL 2 — Load Your HuggingFace Token
# MedGemma is a gated model — you need a HuggingFace account and token to download it.
# This cell reads your token from Kaggle Secrets (the secure way to store passwords/keys).
#
# How to add your token to Kaggle Secrets:
#   1. Top menu → Add-ons → Secrets
#   2. Click "+ Add a New Secret"
#   3. Name: HF_TOKEN  |  Value: your token from huggingface.co
#   4. Toggle "Notebook access" ON (it must turn blue)
#
# If Secrets aren't set up, paste your token directly into HF_TOKEN below as a fallback.

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('✅ HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    HF_TOKEN = 'hf_your_token_here'  # Fallback: paste your token here
    print(f'⚠️  Kaggle Secrets not available ({e})')
    print('   Paste your HuggingFace token directly into HF_TOKEN above.')

if HF_TOKEN and HF_TOKEN != 'hf_your_token_here':
    print(f'   Token loaded — first 8 chars: {HF_TOKEN[:8]}...')
else:
    print('❌ HF_TOKEN is not set — model download will fail')

In [ ]:
# CELL 3 — Check GPU
# This confirms that a GPU is active before we try to load the model.
# MedGemma is a 4 billion parameter model — it needs a GPU to run in reasonable time.
# On a T4 GPU (16GB VRAM), inference takes ~10 seconds per image.
# On CPU, the same task would take 30–90 minutes — essentially unusable.
#
# If you see ❌ No GPU: go to Settings (right panel) → Accelerator → GPU T4 x1
# Do NOT use P100 — it's too old for this model's required PyTorch version.

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU detected: {gpu_name}')
    print(f'   Available VRAM: {vram_gb:.1f} GB')
else:
    print('❌ No GPU found — go to Settings → Accelerator → GPU T4 x1, then restart the kernel')

In [ ]:
# CELL 4 — Download and Load the MedGemma Model
# This downloads Google's MedGemma 1.5 (4B) model from HuggingFace (~8 GB).
# The first time you run this it will take 2–4 minutes to download.
# After that, Kaggle caches it so it loads faster on subsequent runs.
#
# What each setting does:
#   image-text-to-text → tells the pipeline we're sending both an image and text
#   torch_dtype=bfloat16     → loads the model in half precision to save memory
#   device_map='auto'  → automatically places the model on the GPU

import time
from transformers import pipeline

MODEL_ID = 'google/medgemma-1.5-4b-it'

print(f'Loading {MODEL_ID}...')
print('First run downloads ~8 GB — this takes 2–4 minutes. Please wait...')
t0 = time.time()

pipe = pipeline(
    'image-text-to-text',
    model=MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN,
)

print(f'✅ Model ready — loaded in {time.time() - t0:.1f}s')

In [ ]:
# CELL 5 — Load a Sample Chest X-Ray
# Downloads a real chest X-ray from Wikimedia Commons (public domain, free to use).
# This gives us a test image to make sure the model is working before you upload your own.
# The image is displayed so you can see what it looks like before analysis.

import io
import requests
import matplotlib.pyplot as plt
from PIL import Image

SAMPLE_URL = 'https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png'

print('Downloading sample chest X-ray...')
resp = requests.get(SAMPLE_URL, headers={'User-Agent': 'MedGemma-Kaggle/1.0'}, timeout=30)
resp.raise_for_status()
sample_image = Image.open(io.BytesIO(resp.content)).convert('RGB')
print(f'✅ Downloaded — image size: {sample_image.size[0]}x{sample_image.size[1]} pixels')

plt.figure(figsize=(8, 8))
plt.imshow(sample_image, cmap='gray')
plt.title('Sample Chest X-Ray — PA View (Wikimedia Commons, CC0)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 6 — Define the Analysis Function
# This creates a reusable function called analyze_xray() that we use in every analysis cell.
# It works by packaging the image + your question into a chat-style message
# and sending it to MedGemma, which returns a text report.
#
# Parameters:
#   img           → the medical image (PIL Image object)
#   prompt        → the question or instruction (e.g. "generate a radiology report")
#   system_prompt → tells the model what role to play (e.g. radiologist, pathologist)
#
# We also define two default system prompts:
#   SYSTEM_PROMPT_RADIOLOGIST → used for detailed clinical reports
#   SYSTEM_PROMPT_SIMPLE      → used for patient-friendly plain language explanations

from IPython.display import Markdown, display

SYSTEM_PROMPT_RADIOLOGIST = (
    'You are an expert radiologist with over 20 years of clinical experience. '
    'Analyze the provided chest X-ray and generate a comprehensive structured report covering: '
    '1. Lungs  2. Pleura  3. Heart  4. Mediastinum  5. Bones  6. Impression.'
)

SYSTEM_PROMPT_SIMPLE = (
    'You are a compassionate doctor explaining X-ray results to a patient with no '
    'medical background. Use plain, clear language.'
)

def analyze_xray(img, prompt, system_prompt=None):
    # Use the radiologist prompt by default unless a custom one is passed in
    sys_text = system_prompt or SYSTEM_PROMPT_RADIOLOGIST

    # Build the message in chat format: system role + user message with image and question
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': sys_text}]},
        {'role': 'user',   'content': [
            {'type': 'image', 'image': img},   # the medical image
            {'type': 'text',  'text': prompt}, # the question/instruction
        ]},
    ]

    t0 = time.time()
    result = pipe(text=messages, max_new_tokens=1024, do_sample=False)
    # Extract just the model's reply from the full conversation output
    report = result[0]['generated_text'][-1]['content']
    print(f'Done in {time.time() - t0:.1f}s')
    return report

print('✅ analyze_xray() is ready to use')

In [ ]:
# CELL 7 — Analysis 1: Detailed Radiology Report
# Sends the sample X-ray to MedGemma and asks for a full structured report.
# The report covers all major sections a radiologist would include:
# Lungs, Pleura, Heart, Mediastinum, Bones, and a final Impression.
# This takes about 10 seconds on a T4 GPU.

print('Generating detailed radiology report...')
report = analyze_xray(
    sample_image,
    'Generate a detailed radiology report for this chest X-ray.'
)
display(Markdown('## 📋 Detailed Radiology Report\n\n' + report))
display(Markdown('---\n> ⚠️ *For research purposes only. Not for clinical diagnosis.*'))

In [ ]:
# CELL 8 — Analysis 2: Key Findings as Bullet Points
# Instead of a full report, this asks MedGemma to summarize just the important findings.
# Useful when you want a quick overview without reading through a full paragraph report.

print('Extracting key findings...')
findings = analyze_xray(
    sample_image,
    'List the key findings in this chest X-ray as bullet points.'
)
display(Markdown('## 📌 Key Findings\n\n' + findings))

In [ ]:
# CELL 9 — Analysis 3: Patient-Friendly Explanation
# This uses a different system prompt — instead of acting as a radiologist,
# MedGemma is told to act as a doctor explaining results to a regular patient.
# The output avoids medical jargon and uses plain, easy-to-understand language.
# Great for sharing results with someone who has no medical background.

print('Generating patient-friendly explanation...')
simple = analyze_xray(
    sample_image,
    'Explain what you see in this X-ray in simple terms a patient could understand.',
    system_prompt=SYSTEM_PROMPT_SIMPLE
)
display(Markdown('## 💬 Patient-Friendly Explanation\n\n' + simple))

In [ ]:
# CELL 10 — Analyze YOUR OWN Medical Image
# Use this cell to analyze any medical image you upload to Kaggle.
# Supports: chest X-ray, CT scan, MRI, mammogram, skin lesion,
#           pathology slide, retinal fundus, ultrasound, bone X-ray
#
# HOW TO UPLOAD YOUR IMAGE:
#   1. Right panel → Input → Upload → New Dataset
#   2. Drag and drop your image file (PNG or JPG)
#   3. Give the dataset a simple name (e.g. "my-xray")
#   4. Run this cell to find the exact file path:
#
#      import os
#      for root, dirs, files in os.walk('/kaggle/input'):
#          for f in files:
#              print(os.path.join(root, f))
#
#   5. Copy the printed path and paste it into IMAGE_PATH below

from PIL import Image
import os
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# ── STEP 1: Paste your image path here ───────────────────────────────────────
IMAGE_PATH = '/kaggle/input/my-xray/chest.png'  # <-- update this

# ── STEP 2: Uncomment the line that matches your image type ──────────────────
# Each type uses a different prompt and specialist role for the best results.
# Benchmark accuracies are from Google's official MedGemma 1.5 model card.

IMAGE_TYPE = 'chest_xray'       # Chest X-ray        — 88.9% macro F1 (MIMIC)
# IMAGE_TYPE = 'ct_scan'        # CT scan slice       — 61.1% accuracy (7 conditions)
# IMAGE_TYPE = 'mri_brain'      # Brain MRI           — neuro specialist prompt
# IMAGE_TYPE = 'mri_other'      # Other MRI           — 64.7% accuracy (10 conditions)
# IMAGE_TYPE = 'mammogram'      # Mammogram           — BI-RADS assessment
# IMAGE_TYPE = 'skin_lesion'    # Skin lesion         — ABCDE criteria evaluation
# IMAGE_TYPE = 'pathology_slide'# Pathology/biopsy    — 70.0% PathMCQA, 94.5% CRC100k
# IMAGE_TYPE = 'fundus'         # Retinal fundus      — 76.8% accuracy (EyePACS)
# IMAGE_TYPE = 'ultrasound'     # Ultrasound          — organ/mass description
# IMAGE_TYPE = 'bone_xray'      # Bone/joint X-ray    — fracture and lesion analysis

# Each image type maps to a specific clinical question and specialist role
PROMPTS = {
    'chest_xray':      'Generate a detailed radiology report for this chest X-ray covering lungs, pleura, heart, mediastinum, bones, and impression.',
    'ct_scan':         'Analyze this CT scan slice. Describe all visible structures, any masses, nodules, lesions, or abnormalities, and provide an impression.',
    'mri_brain':       'Generate a detailed radiology report for this brain MRI. Cover brain parenchyma, ventricles, sulci, white matter, any masses or lesions, midline shift, and impression.',
    'mri_other':       'Analyze this MRI image. Describe the anatomical structures visible, signal intensities, any abnormal findings, and provide a clinical impression.',
    'mammogram':       'Analyze this mammogram. Describe breast density, any masses, calcifications, or architectural distortions. Provide a BI-RADS style assessment.',
    'skin_lesion':     'Evaluate this skin lesion using ABCDE criteria (Asymmetry, Border, Color, Diameter, Evolution). Describe morphology and provide a differential diagnosis.',
    'pathology_slide': 'Analyze this histopathology slide. Describe cell morphology, tissue architecture, staining patterns, and any features suggestive of malignancy or other pathology.',
    'fundus':          'Analyze this fundus photograph. Describe the optic disc, macula, retinal vessels, and any pathological findings such as hemorrhages, exudates, or neovascularization.',
    'ultrasound':      'Analyze this ultrasound image. Describe the visible organ(s), echogenicity, any masses or cystic lesions, and provide an impression.',
    'bone_xray':       'Analyze this bone X-ray. Describe cortical integrity, bone density, joint spaces, any fractures, lytic or blastic lesions, and provide an impression.',
}

SYSTEM_PROMPTS = {
    'chest_xray':      'You are an expert radiologist with 20 years of experience in chest imaging.',
    'ct_scan':         'You are an expert radiologist specializing in CT interpretation.',
    'mri_brain':       'You are an expert neuroradiologist with 20 years of experience.',
    'mri_other':       'You are an expert radiologist specializing in MRI interpretation.',
    'mammogram':       'You are an expert breast radiologist with 20 years of mammography experience.',
    'skin_lesion':     'You are an expert dermatologist and dermoscopist with 20 years of clinical experience.',
    'pathology_slide': 'You are an expert pathologist with 20 years of experience in histopathology and oncology.',
    'fundus':          'You are an expert ophthalmologist specializing in retinal imaging.',
    'ultrasound':      'You are an expert radiologist specializing in diagnostic ultrasound.',
    'bone_xray':       'You are an expert musculoskeletal radiologist with 20 years of experience.',
}

# ── Load image, display it, and run analysis ─────────────────────────────────
if os.path.exists(IMAGE_PATH):
    custom_image = Image.open(IMAGE_PATH).convert('RGB')
    print(f'✅ Image loaded: {IMAGE_PATH}')
    print(f'   Size: {custom_image.size[0]}x{custom_image.size[1]} pixels')
    print(f'   Type: {IMAGE_TYPE}')

    # Show the image before analysis
    plt.figure(figsize=(7, 7))
    plt.imshow(custom_image, cmap='gray')
    plt.title(os.path.basename(IMAGE_PATH))
    plt.axis('off')
    plt.show()

    # Pick the right prompt and specialist role for this image type
    prompt        = PROMPTS.get(IMAGE_TYPE, PROMPTS['chest_xray'])
    system_prompt = SYSTEM_PROMPTS.get(IMAGE_TYPE, SYSTEM_PROMPTS['chest_xray'])

    print('\nGenerating report — this takes ~10 seconds...')
    custom_report = analyze_xray(custom_image, prompt, system_prompt=system_prompt)

    display(Markdown(f'## 📋 Report: {os.path.basename(IMAGE_PATH)}\n\n' + custom_report))
    display(Markdown('---\n> ⚠️ *For research purposes only. Not for clinical diagnosis.*'))

else:
    print(f'⚠️  Image not found at: {IMAGE_PATH}')
    print()
    print('Run the following to find your exact file path:')
    print('   import os')
    print('   for root, dirs, files in os.walk("/kaggle/input"):')
    print('       for f in files:')
    print('           print(os.path.join(root, f))')